In [ ]:
%matplotlib inline
import torch
import random
import matplotlib.pyplot as plt

In [ ]:
def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = linreg(X, w, b)
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

def data_iter(batch_size, features, labels):
    """生成一个小批量"""
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)  # 样本的读取顺序是随机的
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)]
        )
        yield features[batch_indices], labels[batch_indices]

def linreg(X, w, b):
    """线性回归模型"""
    return torch.matmul(X, w) + b

def squared_loss(y_hat, y):
    """计算均方损失"""
    return ((y_hat-y.reshape(y_hat.shape))**2)/2

def sgd(params, lr, batch_size):
    """计算参数的梯度，并更新参数"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)


In [ ]:
lr = 0.2
num_epochs = 10
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
for epoch in range(num_epochs):
    for X, y in data_iter(10, features, labels):
        l = squared_loss(linreg(X, w, b), y)
        l.sum().backward()
        sgd([w,b], lr, batch_size=10)
    with torch.no_grad():
        train_l = squared_loss(linreg(features, w, b), labels)
        print('epoch', epoch + 1, 'loss', float(train_l.mean()))
with torch.no_grad():
    print(f'估计的w差值: {w.reshape(true_w.shape)-true_w}')
    print(f'估计的b差值: {b - true_b}')
